In [1]:
!pip install langchain langchain-community faiss-cpu sentence-transformers transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import files
uploaded = files.upload()  # upload your Mental_Health_FAQ.csv again

import pandas as pd
df = pd.read_csv("Mental_Health_FAQ.csv")
print(f"Loaded {len(df)} rows")

Saving Mental_Health_FAQ.csv to Mental_Health_FAQ.csv
Loaded 98 rows


In [4]:
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Convert dataset into documents
docs = []
for _, row in df.iterrows():
    content = f"Question: {row['Questions']}\nAnswer: {row['Answers']}"
    docs.append(Document(page_content=content))

print(f"Created {len(docs)} documents")

# Load embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Build FAISS vector store
vectorstore = FAISS.from_documents(docs, embeddings)
print("Vector store built!")

/tmp/ipykernel_3096/1117631179.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


Created 98 documents


/tmp/ipykernel_3096/1117631179.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store built!


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load your fine-tuned model from HuggingFace
model_name = "hamzaN1/mental-health-chatbot"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def ask(question, k=3):
    # Step 1: Retrieve top-k relevant docs
    results = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join([r.page_content for r in results])

    # Step 2: Build prompt with context
    prompt = f"""You are a mental health assistant. Use the context below to answer the question accurately.

Context:
{context}

Question: {question}
Answer:"""

    # Step 3: Generate answer
    input_ids = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).input_ids
    outputs = model.generate(input_ids, max_length=300, no_repeat_ngram_size=3, num_beams=4)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

print("RAG pipeline ready!")

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.38k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

RAG pipeline ready!


In [6]:
questions = [
    "What causes mental illness?",
    "How can I help someone with depression?",
    "What is anxiety disorder?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("-" * 50)

Q: What causes mental illness?
A: 1 in 5 adults in America, and that 1 in 24 adults have a serious mental illness.
--------------------------------------------------
Q: How can I help someone with depression?
A: Talk about it. Listen actively. Listen to what they have to say. Let them tell you what they’re ready to talk about.
--------------------------------------------------
Q: What is anxiety disorder?
A: Anxiety may come up unexpectedly, for seemingly no reason. The anxiety response to a situation or problem may be much stronger than they would expect.
--------------------------------------------------


In [7]:
def ask(question, k=1):
    results = vectorstore.similarity_search(question, k=k)
    context = results[0].page_content  # top match

    # Extract just the Answer part from the retrieved doc
    if "Answer:" in context:
        direct_answer = context.split("Answer:")[1].strip()
    else:
        direct_answer = context

    return direct_answer

# Test again
questions = [
    "What causes mental illness?",
    "How can I help someone with depression?",
    "What is anxiety disorder?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("-" * 50)

Q: What causes mental illness?
A: It is estimated that mental illness affects 1 in 5 adults in America, and that 1 in 24 adults have a serious mental illness. Mental illness does not discriminate; it can affect anyone, regardless of gender, age, income, social status, ethnicity, religion, sexual orientation, or background. Although mental illness can affect anyone, certain conditions may be more common in different populations. For instance, eating disorders tend to occur more often in females, while disorders such as attention deficit/hyperactivity disorder is more prevalent in children. Additionally, all ages are susceptible, but the young and the old are especially vulnerable. Mental illnesses usually strike individuals in the prime of their lives, with 75 percent of mental health conditions developing by the age of 24. This makes identification and treatment of mental disorders particularly difficult, because the normal personality and behavioral changes of adolescence may mask sym